# Exercise 02: Regression — Solution

Complete reference solution for the regression exercise. Your exact numbers may differ slightly depending on library versions, but the patterns should be the same.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
DATA_DIR = "../../../../resources/datasets"

## Part 1: Prepare the Data

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/car-sales-extended.csv")
df.head()

In [ ]:
# Separate features and target
X = df.drop("Price", axis=1)
y = df["Price"]

# One-hot encode categorical columns
X = pd.get_dummies(X, columns=["Make", "Colour"])

print(f"Features shape: {X.shape}")
print(f"Columns: {list(X.columns)}")

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## Part 2: Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

print(f"Linear Regression Results:")
print(f"  MAE:  ${lr_mae:,.2f}")
print(f"  RMSE: ${lr_rmse:,.2f}")
print(f"  R²:   {lr_r2:.4f}")

**Interpretation:** The MAE tells us how far off our predictions are on average. For a car pricing model, being off by a few thousand dollars on a $15-40K car is decent for a first attempt but probably not production-ready.

The R² tells us what proportion of the variance in prices our model explains. An R² around 0.3-0.5 means we're capturing some of the pattern but missing a lot — which makes sense, since Make, Colour, Odometer, and Doors don't capture everything that determines a car's price (year, trim level, condition, features, etc.).

## Part 3: Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print(f"Random Forest Results:")
print(f"  MAE:  ${rf_mae:,.2f}")
print(f"  RMSE: ${rf_rmse:,.2f}")
print(f"  R²:   {rf_r2:.4f}")

print(f"\nComparison:")
print(f"  MAE improvement: ${lr_mae - rf_mae:,.2f} ({(lr_mae - rf_mae) / lr_mae:.1%})")
print(f"  R² improvement:  {rf_r2 - lr_r2:.4f}")

In [ ]:
# Feature importances
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_importance.plot(kind="barh")
plt.title("Feature Importances — Random Forest Regressor")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# Odometer is by far the most important predictor, which makes intuitive sense:
# higher mileage = lower price. Make matters too (BMW vs Honda).
# Colour and Doors have relatively low importance.

## Part 4: Cross-Validation

In [ ]:
lr_cv_scores = cross_val_score(
    LinearRegression(), X, y, cv=5, scoring="neg_mean_absolute_error"
)
rf_cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=100, random_state=42),
    X, y, cv=5, scoring="neg_mean_absolute_error"
)

print(f"Linear Regression CV MAE: ${-lr_cv_scores.mean():,.2f} (±${lr_cv_scores.std():,.2f})")
print(f"Random Forest CV MAE:     ${-rf_cv_scores.mean():,.2f} (±${rf_cv_scores.std():,.2f})")

# The cross-validation results should confirm what we saw on the single split:
# Random Forest typically outperforms Linear Regression on this dataset.
# The ± values show how much the score varies across folds — lower is better.

**Recommendation:** Random Forest wins on accuracy, but consider the tradeoff: Linear Regression is simpler, more interpretable ("each additional km on the odometer reduces price by $X"), and faster. For a production system, you'd want to weigh accuracy against interpretability based on the client's needs.

If the client values transparency ("why did the model price this car at $15K?"), Linear Regression might be preferable despite lower accuracy. If they just want the best predictions, go with Random Forest.

## Part 5: Consulting Scenario

**Client email draft:**

> Our model can estimate trade-in values based on the vehicle's make, mileage, color, and door configuration. On average, the estimate is within about $X,XXX of the actual sale price. For a ballpark quote at the counter, this could save your team significant time.
>
> That said, this model only uses four data points about each vehicle. In reality, factors like model year, trim level, condition, and local market demand also affect price significantly. We'd recommend using this as a starting point for your adjusters, not as the final number — think of it as a first draft that gets you 70% of the way there.
>
> To improve accuracy, we'd want to incorporate additional vehicle data (year, trim, condition rating). With those additions, we'd expect to get the estimates much closer to actual sale prices.

Notice: no mention of MAE, RMSE, R², or any technical terms. The client cares about "how far off" and "can we use it" — not the metric names.